# LoRA Soup: plain LoRA vs AdaLoRA vs L1RA (step 1000)

**Гипотеза:** суп из адаптеров, обученных адаптивными методами (AdaLoRA, L1RA),
даёт лучший результат чем суп из обычных LoRA, потому что адаптивные методы
к шагу 1000 уже нашли «важные» направления в пространстве весов.

**Схема:**
```
для каждого метода в {lora_r16, adalora, l1ra}:
    ΔW_soup = (ΔW_meta_math + ΔW_open_orca) / 2   # где ΔW = B @ A * (alpha/r)
    W_merged = W_base + ΔW_soup
    val_PPL = evaluate(W_merged, val_meta_math + val_open_orca)
```

**Результат:** таблица val PPL для трёх методов + baseline (базовая модель без адаптера).

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/diploma/project

## 1. Install dependencies

In [ ]:
!pip install -q safetensors transformers datasets accelerate

## 2. Imports & device

In [ ]:
import os, gc, math, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from safetensors.torch import load_file
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'Device : {DEVICE}')
print(f'Dtype  : {DTYPE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 3. Config — пути к адаптерам

In [ ]:

BASE_DIR   = Path('.')
LOGS_DIR   = BASE_DIR / 'logs_v2'
DATA_DIR   = BASE_DIR / 'data'
MODEL_NAME = 'Qwen/Qwen3-0.6B'

ADAPTERS = {
    'lora_r16': {
        'meta_math' : LOGS_DIR / 'meta_math_lora_r16'  / 'checkpoints' / 'step_1000',
        'open_orca' : LOGS_DIR / 'open_orca_lora_r16'  / 'checkpoints' / 'step_1000',
    },
    'adalora': {
        'meta_math' : LOGS_DIR / 'meta_math_adalora_init32_target16' / 'checkpoints' / 'step_1000',
        'open_orca' : LOGS_DIR / 'open_orca_adalora_init32_target16' / 'checkpoints' / 'step_1000',
    },
    'l1ra': {
        'meta_math' : LOGS_DIR / 'meta_math_l1ra_r32_lam0.0003' / 'checkpoints' / 'step_1000',
        'open_orca' : LOGS_DIR / 'open_orca_l1ra_r32_lam0.0003' / 'checkpoints' / 'step_1000',
    },
}

VAL_SIZE   = 500
MAX_LEN    = 512
BATCH_SIZE = 4
N_RAW      = 50_000

print('Checking adapter paths...')
all_ok = True
for method, datasets in ADAPTERS.items():
    for ds, path in datasets.items():
        sf = path / 'adapter_model.safetensors'
        ok = sf.exists()
        status = '✓' if ok else '✗ MISSING'
        print(f'  [{status}] {method}/{ds}: {sf}')
        if not ok:
            all_ok = False
print()
print('All paths OK!' if all_ok else '⚠ Some paths are missing — check config above')

## 4. Загрузка токенайзера и валидационных данных

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('Tokenizer loaded:', MODEL_NAME)

In [ ]:
def load_val_data(n_raw=N_RAW, val_size=VAL_SIZE, seed=42):
    """Загружает validation split для оценки LoRA soup."""
    rng = np.random.default_rng(seed)

    mm = (pd.read_parquet(f'{DATA_DIR}/meta-math_MetaMathQA_{n_raw}.parquet')
            .rename(columns={'query': 'question', 'response': 'answer'})[['question','answer']])
    oo = (pd.read_parquet(f'{DATA_DIR}/Open-Orca_OpenOrca_{n_raw}.parquet')
            [['question','response']].rename(columns={'response':'answer'}))

    mm_val = mm.iloc[40_000:40_000+val_size].reset_index(drop=True)
    oo_val = oo.iloc[40_000:40_000+val_size].reset_index(drop=True)

    combined = pd.concat([mm_val, oo_val], ignore_index=True)
    combined = combined.sample(frac=1, random_state=seed).reset_index(drop=True)
    print(f'Val set: {len(combined)} examples ({val_size} from each dataset)')
    return combined

val_df = load_val_data()
val_df.head(3)

## 5. Dataset & DataLoader

In [ ]:
class SoupValDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=MAX_LEN):
        self.samples = []
        for _, row in df.iterrows():
            text = f"Question: {row['question']}\nAnswer: {row['answer']}{tokenizer.eos_token}"
            enc  = tokenizer(text, truncation=True, max_length=max_length,
                             return_tensors='pt')
            ids  = enc['input_ids'].squeeze(0)
            if len(ids) >= 4:
                self.samples.append(ids)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    max_len = max(x.size(0) for x in batch)
    input_ids      = torch.zeros(len(batch), max_len, dtype=torch.long)
    attention_mask = torch.zeros(len(batch), max_len, dtype=torch.long)
    labels         = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, x in enumerate(batch):
        L = x.size(0)
        input_ids[i, :L]      = x
        attention_mask[i, :L] = 1
        labels[i, :L]         = x
    return {'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels}

val_ds     = SoupValDataset(val_df, tokenizer)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=2, pin_memory=True)
print(f'Val dataset: {len(val_ds)} samples, {len(val_loader)} batches')

## 6. Функция оценки PPL

In [ ]:
@torch.no_grad()
def evaluate_ppl(model, loader, device=DEVICE):
    """Считает perplexity на validation split."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out   = model(**batch)
        loss  = out.loss
        n_tok = (batch['labels'] != -100).sum().item()
        total_loss   += loss.item() * n_tok
        total_tokens += n_tok
    avg_loss = total_loss / total_tokens
    ppl      = math.exp(avg_loss)
    return ppl

## 7. LoRA Soup — ядро алгоритма

In [ ]:
def load_delta_weights(adapter_dir: Path) -> dict:
    """
    Загружает adapter_model.safetensors и возвращает словарь
    {layer_name: ΔW}, где ΔW = B @ A * (alpha / r).

    Имена ключей в safetensors:
      base_model.model.model.layers.N.self_attn.q_proj.lora_A.weight  -> A матрица
      base_model.model.model.layers.N.self_attn.q_proj.lora_B.weight  -> B матрица
    """
    cfg_path = adapter_dir / 'adapter_config.json'
    with open(cfg_path) as f:
        cfg = json.load(f)

    default_alpha = cfg.get('lora_alpha', 32)
    default_r     = cfg.get('r', 16)
    alpha_pattern = cfg.get('alpha_pattern', {}) or {}
    rank_pattern  = cfg.get('rank_pattern',  {}) or {}

    st = load_file(adapter_dir / 'adapter_model.safetensors', device='cpu')

    layer_keys = {}
    for key in st.keys():
        if 'lora_A' in key:
            prefix = key.replace('.lora_A.weight', '')
            layer_keys.setdefault(prefix, {})['A'] = key
        elif 'lora_B' in key:
            prefix = key.replace('.lora_B.weight', '')
            layer_keys.setdefault(prefix, {})['B'] = key

    delta_weights = {}
    for prefix, ab in layer_keys.items():
        if 'A' not in ab or 'B' not in ab:
            continue
        A = st[ab['A']].float()
        B = st[ab['B']].float()

        short = prefix.split('.')[-1]
        r     = rank_pattern.get(short,  rank_pattern.get(prefix,  default_r))
        alpha = alpha_pattern.get(short, alpha_pattern.get(prefix, default_alpha))
        if r == 0:
            continue
        scaling = alpha / r

        dW = (B @ A) * scaling
        delta_weights[prefix] = dW

    return delta_weights


def soup_delta_weights(dw1: dict, dw2: dict) -> dict:
    """Усредняет две коллекции ΔW."""
    keys = set(dw1.keys()) & set(dw2.keys())
    if len(keys) != len(dw1) or len(keys) != len(dw2):
        print(f'  Предупреждение: разные ключи, dw1={len(dw1)}, dw2={len(dw2)}, common={len(keys)}')
    result = {}
    for k in keys:
        result[k] = (dw1[k] + dw2[k]) / 2.0
    return result


def apply_soup_to_model(base_model, soup: dict):
    """Прибавляет ΔW_soup к весам базовой модели."""
    sd = base_model.state_dict()
    applied = 0
    for prefix, dW in soup.items():
        if prefix.startswith('base_model.model.'):
            param_key = prefix[len('base_model.model.'):] + '.weight'
        else:
            param_key = prefix + '.weight'

        if param_key not in sd:
            print(f'  Предупреждение: {param_key} не найден в state_dict модели')
            continue

        sd[param_key] = sd[param_key] + dW.to(sd[param_key].dtype).to(sd[param_key].device)
        applied += 1

    base_model.load_state_dict(sd)
    print(f'  Applied {applied} ΔW matrices to model')
    return base_model

print('LoRA Soup functions defined ✓')

## 8. Загрузка базовой модели

In [ ]:
print(f'Loading base model: {MODEL_NAME}')
print('(это займёт ~1-2 минуты...)')

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map='auto',
    trust_remote_code=True,
)
base_model.eval()
print(f'Model loaded. Params: {sum(p.numel() for p in base_model.parameters())/1e6:.1f}M')

## 9. Baseline: PPL базовой модели (без адаптеров)

In [ ]:
import copy

print('Evaluating baseline (no adapter)...')
baseline_ppl = evaluate_ppl(base_model, val_loader)
print(f'Baseline PPL: {baseline_ppl:.4f}')

## 10. Основной эксперимент: LoRA Soup для трёх методов

In [ ]:
results = {'baseline': baseline_ppl}

for method, datasets in ADAPTERS.items():
    print(f'\n{"="*50}')
    print(f'Method: {method}')
    print(f'{"="*50}')

    print(f'  Loading ΔW from meta_math ({datasets["meta_math"].name})...')
    dw_mm = load_delta_weights(datasets['meta_math'])
    print(f'  → {len(dw_mm)} layers loaded')

    print(f'  Loading ΔW from open_orca ({datasets["open_orca"].name})...')
    dw_oo = load_delta_weights(datasets['open_orca'])
    print(f'  → {len(dw_oo)} layers loaded')

    print(f'  Computing soup (averaging ΔW)...')
    dw_soup = soup_delta_weights(dw_mm, dw_oo)
    print(f'  → soup has {len(dw_soup)} layers')

    print(f'  Applying soup to model...')
    model_copy = copy.deepcopy(base_model)
    model_copy = apply_soup_to_model(model_copy, dw_soup)

    print(f'  Evaluating PPL...')
    ppl = evaluate_ppl(model_copy, val_loader)
    print(f'  ✓ {method} soup PPL: {ppl:.4f}')

    results[method] = ppl

    del model_copy, dw_mm, dw_oo, dw_soup
    gc.collect()
    torch.cuda.empty_cache()

print('\nDone!')

## 11. Результаты

In [ ]:
import pandas as pd

rows = []
for name, ppl in results.items():
    baseline = results['baseline']
    if name == 'baseline':
        delta = 0.0
        label = '—'
    else:
        delta = ppl - baseline
        label = f'{delta:+.4f}'
    rows.append({
        'method'         : name,
        'val_PPL'        : round(ppl, 4),
        'ΔPPL vs baseline': label,
    })

df_results = pd.DataFrame(rows)
df_results = df_results.sort_values('val_PPL').reset_index(drop=True)

print('\n' + '='*45)
print('        LoRA Soup Results (step 1000)       ')
print('='*45)
print(df_results.to_string(index=False))
print('='*45)
print('Lower PPL = better')

## 12. Сохранение результатов

In [ ]:
import datetime

ts  = datetime.datetime.now().strftime('%Y%m%d_%H%M')
out = BASE_DIR / f'lora_soup_results_{ts}.csv'
df_results.to_csv(out, index=False)
print(f'Results saved to: {out}')
print(df_results)

## 13. График

In [ ]:
import matplotlib.pyplot as plt

methods = df_results['method'].tolist()
ppls    = df_results['val_PPL'].tolist()
colors  = ['#cccccc' if m == 'baseline' else
           '#4c72b0' if m == 'lora_r16' else
           '#dd8452' if m == 'adalora'  else
           '#55a868'
           for m in methods]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(methods, ppls, color=colors, edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=10)
ax.set_ylabel('Validation PPL (lower = better)')
ax.set_title('LoRA Soup @ step 1000: plain LoRA vs AdaLoRA vs L1RA')
ax.set_ylim(min(ppls) * 0.95, max(ppls) * 1.05)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(BASE_DIR / f'lora_soup_plot_{ts}.png', dpi=150)
plt.show()
print('Plot saved.')